# 🔄 Reflexion Agent — Self-Improving AI with LangGraph

This notebook demonstrates the **Reflexion** agentic pattern — an AI agent that doesn't just answer questions, but **critiques its own output**, searches for better evidence, and **iteratively revises** its answer.

## What is a Reflexion Agent?

Unlike a simple ReAct (Reasoning + Acting) agent that just thinks and acts, a Reflexion agent adds a **self-evaluation loop**:

1. **Respond** — Generate an initial answer with structured self-reflection
2. **Execute Tools** — Search for information based on what the reflection identified as missing
3. **Revise** — Improve the answer using new evidence + self-critique
4. **Loop** — Repeat until the answer is satisfactory or max iterations reached

## Tech Stack
- **LLM**: OpenAI GPT-4.1-nano
- **Search**: Tavily Search API
- **Orchestration**: LangGraph (MessageGraph)
- **Structured Output**: Pydantic models for answer, reflection, and revision

In [ ]:
%%capture
%pip install langchain-openai==0.3.10
%pip install langchain==0.3.21
%pip install openai==1.68.2
%pip install langchain-community==0.3.24
%pip install --upgrade langgraph
%pip install python-dotenv

## 1. Imports & Configuration

In [ ]:
import os
import json
from typing import List
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import END, MessageGraph

# Load API keys from .env file (never hardcode secrets!)
load_dotenv()

TAVILY_API_KEY = os.getenv("Tavily_api_key")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("✅ All libraries imported")
print(f"🔑 Tavily API: {'Loaded' if TAVILY_API_KEY else 'Missing'}")
print(f"🔑 OpenAI API: {'Loaded' if OPENAI_API_KEY else 'Missing'}")

## 2. Initialize LLM & Search Tool

In [ ]:
llm = ChatOpenAI(model="gpt-4.1-nano", api_key=OPENAI_API_KEY)
tavily_tool = TavilySearchResults(max_results=3, tavily_api_key=TAVILY_API_KEY)

# Quick test — verify the search tool works
sample_results = tavily_tool.invoke("healthy breakfast recipes")
print(f"✅ Search tool working — got {len(sample_results)} results")

## 3. Define Structured Output Models

The key to Reflexion is **structured self-critique**. The LLM doesn't just answer — it also outputs:
- **Reflection**: What's missing? What's unnecessary?
- **Search Queries**: What to research next to improve the answer

In [ ]:
class Reflection(BaseModel):
    missing: str = Field(description="What information is missing")
    superfluous: str = Field(description="What information is unnecessary")

class AnswerQuestion(BaseModel):
    """Initial answer with self-reflection."""
    answer: str = Field(description="Main response to the question (~250 words)")
    reflection: Reflection = Field(description="Self-critique of the answer")
    search_queries: List[str] = Field(description="1-3 queries for additional research")

class ReviseAnswer(AnswerQuestion):
    """Revised answer incorporating new evidence."""
    references: List[str] = Field(description="Citations supporting the updated answer")

print("✅ Structured output models defined")

## 4. Create the Prompt Template & Chains

We create two chains:
- **Initial Chain** — generates the first answer + reflection
- **Revisor Chain** — revises using search results + previous critique

In [ ]:
prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful research assistant that provides thorough, evidence-based answers.

Your response must follow these steps:
1. {first_instruction}
2. Present a clear, well-structured answer with evidence and reasoning.
3. Reflect and critique your own answer — identify what's missing and what's unnecessary.
4. After the reflection, list 1-3 search queries for finding additional information.

Focus on accuracy, completeness, and citing sources when available."""
    ),
    MessagesPlaceholder(variable_name="messages"),
    (
        "system",
        "Answer the user's question using the required structured format."
    ),
])

# Initial responder chain
first_responder_prompt = prompt_template.partial(
    first_instruction="Provide a detailed ~250 word answer"
)
initial_chain = first_responder_prompt | llm.bind_tools(tools=[AnswerQuestion])

# Revisor chain
revise_instructions = """Revise your previous answer using the new information.
- Use the previous critique to add missing information and remove unnecessary content.
- Include numerical citations referencing search results.
- Add a References section with URLs from search results.
- Keep response under 250 words, prioritizing precision over volume."""

revisor_prompt = prompt_template.partial(first_instruction=revise_instructions)
revisor_chain = revisor_prompt | llm.bind_tools(tools=[ReviseAnswer])

print("✅ Initial chain and revisor chain created")

## 5. Test the Initial Chain

Let's see the structured output: answer + reflection + search queries.

In [ ]:
question = "Any ideas for a healthy breakfast"
response = initial_chain.invoke({"messages": [HumanMessage(content=question)]})

args = response.tool_calls[0]["args"]

print("📝 INITIAL ANSWER:")
print(args["answer"])
print("\n🪞 SELF-REFLECTION:")
print(f"  Missing: {args['reflection']['missing']}")
print(f"  Superfluous: {args['reflection']['superfluous']}")
print("\n🔍 SEARCH QUERIES:")
for q in args["search_queries"]:
    print(f"  • {q}")

## 6. Define the Tool Execution Node

This node takes the search queries from the LLM's structured output and runs them through Tavily.

In [ ]:
def execute_tools(state: List[BaseMessage]) -> List[ToolMessage]:
    """Execute search queries from the last AI message's structured output."""
    last_ai_message = state[-1]
    tool_messages = []

    for tool_call in last_ai_message.tool_calls:
        if tool_call["name"] in ["AnswerQuestion", "ReviseAnswer"]:
            call_id = tool_call["id"]
            search_queries = tool_call["args"].get("search_queries", [])
            query_results = {}
            for query in search_queries:
                result = tavily_tool.invoke(query)
                query_results[query] = result
            tool_messages.append(
                ToolMessage(content=json.dumps(query_results), tool_call_id=call_id)
            )

    return tool_messages

print("✅ Tool execution node defined")

## 7. Build the Reflexion Graph

This is where it all comes together. The graph flow:

```
respond → execute_tools → revise → (loop back or END)
```

The `event_loop` counts how many tool executions have happened and stops after `MAX_ITERATIONS`.

In [ ]:
MAX_ITERATIONS = 2

def event_loop(state: List[BaseMessage]) -> str:
    """Stop after MAX_ITERATIONS revision cycles."""
    count_tool_visits = sum(isinstance(item, ToolMessage) for item in state)
    if count_tool_visits >= MAX_ITERATIONS:
        return END
    return "execute_tools"

# Build the graph
graph = MessageGraph()

graph.add_node("respond", initial_chain)
graph.add_node("execute_tools", execute_tools)
graph.add_node("revise", revisor_chain)

graph.set_entry_point("respond")
graph.add_edge("respond", "execute_tools")
graph.add_edge("execute_tools", "revise")
graph.add_conditional_edges("revise", event_loop)

app = graph.compile()
print("✅ Reflexion graph compiled")

## 8. Run the Agent

Let's test with a real question and see how the agent improves its answer through reflection.

In [ ]:
test_question = """I'm pre-diabetic and need to lower my blood sugar, and I have heart issues.
What breakfast foods should I eat and avoid?"""

responses = app.invoke(test_question)
print(f"✅ Agent completed — {len(responses)} messages in conversation")

## 9. View Results — Initial Draft vs. Final Revised Answer

This is the power of Reflexion: compare the first draft to the final output after self-critique and research.

In [ ]:
# Extract the initial draft
initial_answer = responses[1].tool_calls[0]["args"]["answer"]
initial_reflection = responses[1].tool_calls[0]["args"]["reflection"]

print("=" * 60)
print("📝 INITIAL DRAFT")
print("=" * 60)
print(initial_answer)
print(f"\n🪞 Reflection — Missing: {initial_reflection['missing']}")
print(f"🪞 Reflection — Superfluous: {initial_reflection['superfluous']}")

In [ ]:
# Extract all revised answers (walking backwards to find the final one)
answers = []
for msg in reversed(responses):
    if getattr(msg, "tool_calls", None):
        for tool_call in msg.tool_calls:
            answer = tool_call.get("args", {}).get("answer")
            refs = tool_call.get("args", {}).get("references", [])
            if answer:
                answers.append({"answer": answer, "references": refs})

print("=" * 60)
print("✅ FINAL REVISED ANSWER")
print("=" * 60)
print(answers[0]["answer"])

if answers[0]["references"]:
    print("\n📚 References:")
    for i, ref in enumerate(answers[0]["references"], 1):
        print(f"  [{i}] {ref}")